In [ ]:
!pip install -q spacy

In [ ]:
!pip install -U spacy[cuda]
!nvcc --version
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of thinc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.0/33.0 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 29.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for cupy
  Running setup.py clean for cupy
  error: subprocess-exited-with-erro

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("jigsaw-toxic-comment-train.csv", on_bad_lines='skip',engine='python')
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223549 entries, 0 to 223548
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             223549 non-null  object
 1   comment_text   223549 non-null  object
 2   toxic          223549 non-null  int64 
 3   severe_toxic   223549 non-null  int64 
 4   obscene        223549 non-null  int64 
 5   threat         223549 non-null  int64 
 6   insult         223549 non-null  int64 
 7   identity_hate  223549 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 13.6+ MB


In [ ]:
df = df.drop(columns = ['id'])

In [ ]:
one_hot_cols = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']

In [ ]:
df['category'] = df[one_hot_cols].idxmax(axis=1)

In [ ]:
df.head()

,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,category
0,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,toxic
1,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,toxic
2,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,toxic
3,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,toxic
4,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,toxic


In [ ]:
df = df.drop(columns = one_hot_cols)

In [ ]:
df

,comment_text,category
0,Explanation\nWhy the edits made under my usern...,toxic
1,D'aww! He matches this background colour I'm s...,toxic
2,"Hey man, I'm really not trying to edit war. It...",toxic
3,"""\nMore\nI can't make any real suggestions on ...",toxic
4,"You, sir, are my hero. Any chance you remember...",toxic
...,...,...
223544,":Jerome, I see you never got around to this…! ...",toxic
223545,==Lucky bastard== \n http://wikimediafoundatio...,toxic
223546,==shame on you all!!!== \n\n You want to speak...,toxic
223547,MEL GIBSON IS A NAZI BITCH WHO MAKES SHITTY MO...,toxic


Text Preprocessing

In [ ]:
import spacy
import re
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])  # Faster by disabling unneeded components
spacy.require_gpu()
# Set of stopwords
stopwords = nlp.Defaults.stop_words

# Simple regex stemmer
stemmer = re.compile(r'(ing$|ed$|s$)')

In [ ]:
def batch_process_spacy(texts):
    results = []
    for doc in nlp.pipe(texts, batch_size=1000, n_process=2):  # Adjust batch and process count
        tokens = [
            stemmer.sub('', token.text.lower())
            for token in doc
            if token.text.lower() not in stopwords and token.is_alpha
        ]
        results.append(" ".join(tokens))
    return results

df['cleaned_comment'] = batch_process_spacy(df['comment_text'].tolist())

In [ ]:
df['cleaned_comment']

,cleaned_comment
0,explanation edit username hardcore metallica f...
1,matche background colour seemingly stuck thank...
2,hey man try edit war guy constantly remov rele...
3,real suggestion improvement wonder section sta...
4,sir hero chance remember page
...,...
223544,jerome got surpris look example nomine plainso...
223545,lucky heh famou kida envy congrat
223546,shame want speak gay romanian
223547,mel gibson nazi bitch make shitty movie buttse...


In [ ]:
df['cleaned_comment'][6]

'cocksucker pis work'

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
df= df[df['cleaned_comment'].notnull() & (df['cleaned_comment'].str.strip() != '')].reset_index(drop=True)


comments = df['cleaned_comment'].astype(str).tolist()

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(comments)
print(f'TF_IDF matrix shape:{tfidf_matrix.shape}')

TF_IDF matrix shape:(223113, 200169)


In [ ]:
from sklearn.decomposition import TruncatedSVD
n_components = 300
svd = TruncatedSVD(n_components=n_components, random_state = 42)
X_reduced = svd.fit_transform(tfidf_matrix)
print(f'After TruncatedSVD,shape :{X_reduced.shape}')

After TruncatedSVD,shape :(223113, 300)


In [ ]:
X_reduced

array([[ 0.15251048,  0.07827054,  0.03897989, ..., -0.0217508 ,
         0.00217676,  0.01218584],
       [ 0.09052287, -0.01230342,  0.1160824 , ..., -0.0027929 ,
        -0.00252753, -0.00283263],
       [ 0.26360956,  0.13708327,  0.02451747, ..., -0.01850448,
        -0.01017487, -0.0088083 ],
       ...,
       [ 0.19076441, -0.07172017,  0.05290423, ..., -0.00058804,
        -0.0240464 , -0.00102167],
       [ 0.01562909, -0.00786032, -0.01398281, ...,  0.00031296,
        -0.03079122,  0.00647125],
       [ 0.05152646, -0.02390117, -0.02763271, ...,  0.00587911,
         0.00960474,  0.00335821]])

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
df['category'] = encoder.fit_transform(df['category'])
len(df['category'])

223113

In [ ]:
np.savez_compressed('hate_comments_reduced.npz', X=X_reduced, y=df['category'])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mv /content/hate_comments_reduced.npz /content/drive/MyDrive/hate_comments_reduced.npz